# Retail Metric View

Creates a Unity Catalog metric view (`retail_q.retail_semantic.retail_metric`) that centralizes retail sales KPIs — revenue, discounts, quantity, order value, customer/product counts — with dimensions from `fact_sales`, `dim_product`, `dim_customer`, and `dim_calendar`.

In [0]:
%sql
-- Create the semantic schema if it does not exist
CREATE SCHEMA IF NOT EXISTS retail_q.retail_semantic
COMMENT 'Semantic layer containing metric views for retail analytics';

In [0]:
%sql
CREATE OR REPLACE VIEW retail_q.retail_semantic.retail_metric
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: retail_q.retail_gold.fact_sales
  comment: >
    Retail metric view centralizing sales KPIs with product, customer,
    and calendar dimensions joined to the fact_sales table.
  joins:
    - name: products
      source: retail_q.retail_gold.dim_product
      on: source.product_id = products.product_id
      cardinality: many_to_one
      rely:
        at_most_one_match: true
    - name: customers
      source: retail_q.retail_gold.dim_customer
      on: source.customer_id = customers.customer_id
      cardinality: many_to_one
      rely:
        at_most_one_match: true
    - name: calendar
      source: retail_q.retail_gold.dim_calendar
      on: source.transaction_date = calendar.date
      cardinality: many_to_one
      rely:
        at_most_one_match: true

  dimensions:
    # ---- Fact-level dimensions ----
    - name: Transaction Date
      expr: source.transaction_date
      display_name: Transaction Date
      comment: Date the sales transaction occurred
      format:
        type: date
        date_format: year_month_day
      synonyms:
        - sales date
        - date

    - name: Transaction Month
      expr: DATE_TRUNC('MONTH', source.transaction_date)
      display_name: Transaction Month
      comment: Calendar month extracted from the transaction date
      format:
        type: date
        date_format: locale_long_month
      synonyms:
        - month
        - sales month

    - name: Payment Mode
      expr: source.payment_mode
      display_name: Payment Mode
      comment: Method of payment (Card, Cash, Crypto, Net Banking, UPI)
      synonyms:
        - payment method
        - payment type

    - name: Sales Channel
      expr: source.sales_channel
      display_name: Sales Channel
      comment: Channel through which the sale occurred (Online or Store)
      synonyms:
        - channel

    - name: Stage Name
      expr: source.stage_name
      display_name: Opportunity Stage
      comment: Sales pipeline stage of the opportunity
      synonyms:
        - opportunity stage
        - pipeline stage

    - name: Store ID
      expr: source.store_id
      display_name: Store ID
      comment: Identifier for the store where the transaction occurred

    - name: Sales Owner ID
      expr: source.owner_id
      display_name: Sales Owner ID
      comment: Identifier of the sales representative who owns the opportunity
      synonyms:
        - owner
        - sales rep

    # ---- Product dimensions ----
    - name: Product Name
      expr: products.product_name
      display_name: Product Name
      comment: Name of the product sold
      synonyms:
        - product

    - name: Category
      expr: products.category
      display_name: Product Category
      comment: Top-level product category (Electronics, Fashion, Groceries, Home)
      synonyms:
        - product category
        - category

    - name: Subcategory
      expr: products.subcategory
      display_name: Product Subcategory
      comment: Product subcategory within the category
      synonyms:
        - sub-category

    - name: Brand
      expr: products.brand
      display_name: Brand
      comment: Product brand name

    - name: Product Segment
      expr: products.product_segment
      display_name: Product Segment
      comment: Product tier (BUDGET, MID_RANGE, PREMIUM)
      synonyms:
        - tier
        - segment

    - name: Supplier Name
      expr: products.supplier_name
      display_name: Supplier Name
      comment: Name of the product supplier
      synonyms:
        - supplier

    # ---- Customer dimensions ----
    - name: Customer Name
      expr: customers.customer_name
      display_name: Customer Name
      comment: Name of the customer
      synonyms:
        - account name
        - customer

    - name: Customer Type
      expr: customers.customer_type
      display_name: Customer Type
      comment: Customer relationship type (Customer - Channel, Customer - Direct)
      synonyms:
        - account type

    - name: Billing City
      expr: customers.billing_city
      display_name: Billing City
      comment: City from the customer's billing address
      synonyms:
        - city

    - name: Billing State
      expr: customers.billing_state
      display_name: Billing State
      comment: State from the customer's billing address
      synonyms:
        - state

    - name: Billing Country
      expr: customers.billing_country
      display_name: Billing Country
      comment: Country from the customer's billing address
      synonyms:
        - country

    - name: Industry
      expr: customers.industry
      display_name: Industry
      comment: Customer's industry sector

    # ---- Calendar dimensions ----
    - name: Calendar Year
      expr: calendar.year
      display_name: Calendar Year
      comment: Year from the calendar dimension
      synonyms:
        - year

    - name: Calendar Quarter
      expr: calendar.year_quarter
      display_name: Calendar Quarter
      comment: Year and quarter (e.g. 2026-Q1)
      synonyms:
        - quarter
        - year quarter

    - name: Calendar Month Name
      expr: calendar.month_name
      display_name: Calendar Month
      comment: Month name from the calendar dimension
      synonyms:
        - month name

    - name: Year Month
      expr: calendar.year_month
      display_name: Year-Month
      comment: Year-month string from the calendar dimension (e.g. 2026-01)
      synonyms:
        - year-month

    - name: Day of Week
      expr: calendar.day_name
      display_name: Day of Week
      comment: Day of the week name from the calendar dimension
      synonyms:
        - weekday

    - name: Is Weekend
      expr: calendar.is_weekend
      display_name: Is Weekend
      comment: Whether the transaction date falls on a weekend

  measures:
    - name: Transaction Count
      expr: COUNT(1)
      display_name: Transaction Count
      comment: Total number of sales transactions
      synonyms:
        - number of transactions
        - order count
        - count

    - name: Total Quantity
      expr: SUM(source.quantity)
      display_name: Total Quantity
      comment: Total units sold across all transactions
      synonyms:
        - units sold
        - total units
      format:
        type: number
        decimal_places:
          type: exact
          places: 0

    - name: Total Revenue
      expr: SUM(source.selling_price)
      display_name: Total Revenue
      comment: Gross revenue from all sales transactions (before discounts)
      synonyms:
        - gross revenue
        - gross sales
        - revenue
        - sales
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2

    - name: Total Discount
      expr: SUM(source.discount_amount)
      display_name: Total Discount
      comment: Total discount amount across all transactions
      synonyms:
        - discount
        - total discounts
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2

    - name: Net Revenue
      expr: SUM(source.selling_price) - SUM(source.discount_amount)
      display_name: Net Revenue
      comment: Revenue after subtracting discounts
      synonyms:
        - net sales
        - revenue net of discount
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2

    - name: Avg Order Value
      expr: MEASURE(`Total Revenue`) / MEASURE(`Transaction Count`)
      display_name: Avg Order Value
      comment: Average revenue per transaction
      synonyms:
        - AOV
        - average order value
        - average sale
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2

    - name: Avg Discount per Transaction
      expr: MEASURE(`Total Discount`) / MEASURE(`Transaction Count`)
      display_name: Avg Discount per Transaction
      comment: Average discount amount per transaction
      synonyms:
        - average discount
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2

    - name: Discount Rate
      expr: MEASURE(`Total Discount`) / MEASURE(`Total Revenue`)
      display_name: Discount Rate
      comment: Proportion of total revenue given as discount
      synonyms:
        - discount percentage
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 2

    - name: Total Opportunity Amount
      expr: SUM(source.amount)
      display_name: Total Opportunity Amount
      comment: Sum of opportunity amounts from the sales pipeline
      synonyms:
        - pipeline value
        - opportunity value
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2

    - name: Won Revenue
      expr: SUM(source.selling_price) FILTER (WHERE `Stage Name` = 'Closed Won')
      display_name: Won Revenue
      comment: Revenue from transactions in the Closed Won stage
      synonyms:
        - closed won revenue
        - won sales
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2

    - name: Unique Customers
      expr: COUNT(DISTINCT source.customer_id)
      display_name: Unique Customers
      comment: Count of distinct customers across transactions
      synonyms:
        - distinct customers
        - customer count
      format:
        type: number
        decimal_places:
          type: exact
          places: 0

    - name: Unique Products
      expr: COUNT(DISTINCT source.product_id)
      display_name: Unique Products
      comment: Count of distinct products sold
      synonyms:
        - distinct products
        - product count
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
$$;

In [0]:
%sql
-- Verify: overall KPIs across the entire dataset
SELECT
  MEASURE(`Transaction Count`)      AS transaction_count,
  MEASURE(`Total Revenue`)         AS total_revenue,
  MEASURE(`Total Discount`)        AS total_discount,
  MEASURE(`Net Revenue`)          AS net_revenue,
  MEASURE(`Avg Order Value`)      AS avg_order_value,
  MEASURE(`Discount Rate`)        AS discount_rate,
  MEASURE(`Won Revenue`)          AS won_revenue,
  MEASURE(`Unique Customers`)     AS unique_customers,
  MEASURE(`Unique Products`)      AS unique_products
FROM retail_q.retail_semantic.retail_metric
GROUP BY ALL;